# Week 2 Day 3 exercise solution

This solution uses three email writers:

- OpenAI with `gpt-5.4-mini`
- Anthropic with `claude-haiku-4-5`
- Google with `gemini-3.1-flash-lite`

A Sales Manager asks all three writers for a draft, selects the best ideas, and returns one structured `SalesEmail`.

The manager has one input guardrail and two output guardrails. The final email is printed only after all guardrails pass.

## Design

The prospect brief is checked before any model runs. The input guardrail is configured with `run_in_parallel=False` so an incomplete brief stops the workflow before the writer tools are called.

The writers return draft text. The OpenAI Sales Manager creates the final Pydantic object because Anthropic's OpenAI-compatible endpoint does not guarantee `response_format` schemas.

The final manager has:

- The course email-quality guardrail for professionalism and placeholders.
- An added claims guardrail that rejects facts not supported by the brief.

Sending is kept outside this notebook. After the guarded run succeeds, the three fields in `SalesEmail` can be passed to the Lab 2 email function.

In [ ]:
from dotenv import load_dotenv
from openai import AsyncOpenAI
from agents import (
    Agent,
    Runner,
    trace,
    OpenAIChatCompletionsModel,
    input_guardrail,
    output_guardrail,
    GuardrailFunctionOutput,
    InputGuardrailTripwireTriggered,
    OutputGuardrailTripwireTriggered,
)
import os
from pydantic import BaseModel, Field

load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
anthropic_api_key = os.getenv("ANTHROPIC_API_KEY")
google_api_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")

provider_keys = {
    "OpenAI": openai_api_key,
    "Anthropic": anthropic_api_key,
    "Google": google_api_key,
}

for provider, api_key in provider_keys.items():
    print(f"{provider} API key is {'set' if api_key else 'not set'}")

In [ ]:
ANTHROPIC_BASE_URL = "https://api.anthropic.com/v1/"
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"

anthropic_client = AsyncOpenAI(
    base_url=ANTHROPIC_BASE_URL,
    api_key=anthropic_api_key or "missing",
)
gemini_client = AsyncOpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=google_api_key or "missing",
)

anthropic_model = OpenAIChatCompletionsModel(
    model="claude-haiku-4-5",
    openai_client=anthropic_client,
)
gemini_model = OpenAIChatCompletionsModel(
    model="gemini-3.1-flash-lite",
    openai_client=gemini_client,
)

## Structured inputs and outputs

In [ ]:
class SalesBrief(BaseModel):
    prospect_company: str = Field(description="The company receiving the email")
    prospect_role: str = Field(description="The role of the person receiving the email")
    business_need: str = Field(description="A relevant problem or goal")
    call_to_action: str = Field(description="The next step requested in the email")
    approved_facts: list[str] = Field(description="Product facts that may be stated")

class SalesEmail(BaseModel):
    subject: str = Field(description="A short subject line")
    text_body: str = Field(description="The plain-text email body")
    html_body: str = Field(description="The same email formatted as simple HTML")

class EmailReview(BaseModel):
    is_professional: bool = Field(description="Whether the email is professional and appropriate")
    contains_placeholders: bool = Field(description="Whether the email contains unfilled placeholders")

class ClaimsReview(BaseModel):
    has_unsupported_claims: bool = Field(description="Whether the email states facts that are not in the approved brief")
    unsupported_claims: list[str] = Field(description="The unsupported claims, or an empty list")

## Guardrails

The input guardrail checks that the brief contains enough information to write a specific email.

The first output guardrail reuses the course checks. The added output guardrail checks that the email does not invent prices, results, guarantees, customer names, or product features.

In [ ]:
email_checker = Agent(
    name="Email Quality Checker",
    instructions=(
        "Review the structured sales email. "
        "Decide whether it is professional and whether it contains unfilled placeholders."
    ),
    model="gpt-5.4-mini",
    output_type=EmailReview,
)

claims_checker = Agent(
    name="Claims Checker",
    instructions=(
        "Compare the sales email with the approved brief. "
        "Mark any price, result, guarantee, customer name, delivery time, or product feature "
        "that is not supported by the approved facts."
    ),
    model="gpt-5.4-mini",
    output_type=ClaimsReview,
)

@input_guardrail(run_in_parallel=False)
async def prospect_brief_guardrail(ctx, agent, agent_input):
    brief = ctx.context
    required_values = {
        "prospect company": brief.prospect_company,
        "prospect role": brief.prospect_role,
        "business need": brief.business_need,
        "call to action": brief.call_to_action,
    }
    missing = [
        name for name, value in required_values.items()
        if not value.strip()
    ]
    if not brief.approved_facts:
        missing.append("approved facts")

    return GuardrailFunctionOutput(
        output_info={"missing_information": missing},
        tripwire_triggered=bool(missing),
    )

@output_guardrail
async def email_quality_guardrail(ctx, agent, email):
    result = await Runner.run(
        email_checker,
        email.model_dump_json(indent=2),
        context=ctx.context,
    )
    review = result.final_output
    is_problem = review.contains_placeholders or not review.is_professional

    return GuardrailFunctionOutput(
        output_info={"review": review},
        tripwire_triggered=is_problem,
    )

@output_guardrail
async def unsupported_claims_guardrail(ctx, agent, email):
    brief = ctx.context
    review_request = f"""
Approved brief:
{brief.model_dump_json(indent=2)}

Email:
{email.model_dump_json(indent=2)}
"""
    result = await Runner.run(
        claims_checker,
        review_request,
        context=brief,
    )
    review = result.final_output

    return GuardrailFunctionOutput(
        output_info={"review": review},
        tripwire_triggered=review.has_unsupported_claims,
    )

## Three provider writers

Each writer receives the same brief and returns one draft. The manager sees the three drafts as tool results.

In [ ]:
writer_instructions = """
You write cold sales emails for ComplAI.
Use only the facts in the prospect brief.
Write a short subject and a concise email body.
Do not use placeholders and do not invent facts.
Return the subject and body as plain text.
"""

openai_writer = Agent(
    name="OpenAI Sales Writer",
    instructions=writer_instructions,
    model="gpt-5.4-mini",
)
anthropic_writer = Agent(
    name="Anthropic Sales Writer",
    instructions=writer_instructions,
    model=anthropic_model,
)
gemini_writer = Agent(
    name="Gemini Sales Writer",
    instructions=writer_instructions,
    model=gemini_model,
)

writer_tool_description = (
    "Write one cold sales email from the supplied prospect brief. "
    "Use only the facts in the brief."
)

writer_tools = [
    openai_writer.as_tool(
        tool_name="openai_email_writer",
        tool_description=writer_tool_description,
    ),
    anthropic_writer.as_tool(
        tool_name="anthropic_email_writer",
        tool_description=writer_tool_description,
    ),
    gemini_writer.as_tool(
        tool_name="gemini_email_writer",
        tool_description=writer_tool_description,
    ),
]

## Sales Manager

The manager calls all three writers, compares their drafts, and returns one `SalesEmail`. It has no sending tool, so no message can be sent before the output guardrails finish.

In [ ]:
manager_instructions = """
You are the Sales Manager at ComplAI.

Use each email-writer tool exactly once and give each tool the complete prospect brief.
Compare the three drafts and prepare one final email using the strongest parts.

Use only the approved facts.
Do not use placeholders.
Keep the text and HTML versions consistent.
Return the final email using the required SalesEmail structure.
"""

sales_manager = Agent(
    name="Sales Manager",
    instructions=manager_instructions,
    tools=writer_tools,
    model="gpt-5.4-mini",
    output_type=SalesEmail,
    input_guardrails=[prospect_brief_guardrail],
    output_guardrails=[
        email_quality_guardrail,
        unsupported_claims_guardrail,
    ],
)

## Run the complete workflow

This brief contains all required information. A successful run prints the structured email after both output guardrails pass.

In [ ]:
brief = SalesBrief(
    prospect_company="Northstar Analytics",
    prospect_role="Head of Security",
    business_need="The team is preparing for its first SOC2 audit and wants a clearer evidence-collection process.",
    call_to_action="Ask whether a 20-minute introductory call next week would be useful.",
    approved_facts=[
        "ComplAI is a SaaS tool for SOC2 compliance.",
        "ComplAI helps teams prepare for audits.",
        "ComplAI uses AI to support the compliance process.",
    ],
)

task = f"""
Create one cold sales email for this prospect.

Prospect brief:
{brief.model_dump_json(indent=2)}
"""

try:
    with trace("Sales email with three providers and guardrails"):
        result = await Runner.run(
            sales_manager,
            task,
            context=brief,
        )

    email = result.final_output
    print(f"Subject: {email.subject}\n")
    print(email.text_body)
    print(f"\nHTML:\n{email.html_body}")

except InputGuardrailTripwireTriggered as exc:
    print(f"Input guardrail stopped the run: {exc.guardrail_result.output.output_info}")

except OutputGuardrailTripwireTriggered as exc:
    print(f"Output guardrail stopped the run: {exc.guardrail_result.output.output_info}")

## Check the input guardrail

This brief is missing a business need. Because the input guardrail is not run in parallel, the workflow stops before any provider model is called.

In [ ]:
incomplete_brief = SalesBrief(
    prospect_company="Northstar Analytics",
    prospect_role="Head of Security",
    business_need="",
    call_to_action="Ask whether a short introductory call would be useful.",
    approved_facts=[
        "ComplAI is a SaaS tool for SOC2 compliance.",
    ],
)

try:
    await Runner.run(
        sales_manager,
        "Write a cold sales email from the supplied brief.",
        context=incomplete_brief,
    )
except InputGuardrailTripwireTriggered as exc:
    print(exc.guardrail_result.output.output_info)